In [ ]:
import pandas as pd

In [ ]:
# Identificadores directos: name, document_id, email, loyalty_card_number
# Cuasi-identificadores: age, city, occupation
# Información sensible/confidencial: annual_spend

raw = pd.read_csv("../data/raw.csv")
raw.head()

In [ ]:
# Información pública externa
auxiliary = pd.read_csv("../data/auxiliary.csv")
auxiliary.head()

In [ ]:
# Anonimización ingenua

anonymized = raw.drop(columns=["name", "document_id", "email", "loyalty_card_number"])
anonymized.head()

In [ ]:
# La combinación de los cuasi-identificadores puede permitir la reidentificación de individuos
quasi_identifiers = ["age", "city", "occupation"]

reidentified = auxiliary.merge(
    anonymized,
    on=quasi_identifiers,
    how="inner",
)

reidentified.head()

In [ ]:
# Paso 1: eliminación de los identificadores directos

anonymized = raw.drop(columns=["name", "email"])
anonymized.head()

In [ ]:
# Paso 2: enmascaramiento del loyalty_card_number

loyalty_card_number = anonymized["loyalty_card_number"].astype(str).str.zfill(12)

anonymized["loyalty_card_number"] = "********" + loyalty_card_number.str[-4:]

anonymized.head()

In [ ]:
# Paso 3: pseudonimización del documento de identidad

import hashlib
import hmac

secret_key = b"clave-secreta-del-programa"


def pseudonymize(document_id):
    digest = hmac.new(
        secret_key,
        str(document_id).encode("utf-8"),
        hashlib.sha256,
    ).hexdigest()

    return f"CUST-{digest[:12].upper()}"


anonymized["customer_id"] = anonymized["document_id"].apply(pseudonymize)

anonymized = anonymized.drop(columns=["document_id"])

anonymized.head()

In [ ]:
# Paso 4: anonimización de la edad

anonymized["age_group"] = pd.cut(
    anonymized["age"],
    bins=[20, 30, 40, 50, 60, 70],
    labels=["20-29", "30-39", "40-49", "50-59", "60-69"],
    right=False,
)

anonymized = anonymized.drop(columns=["age"])

anonymized.head()

In [ ]:
# Verificación

auxiliary_attack = auxiliary.copy()
auxiliary_attack["age_group"] = pd.cut(
    auxiliary_attack["age"],
    bins=[20, 30, 40, 50, 60, 70],
    labels=["20-29", "30-39", "40-49", "50-59", "60-69"],
    right=False,
)

quasi_identifiers = ["age_group", "city", "occupation"]
candidates = auxiliary_attack.merge(
    anonymized,
    on=quasi_identifiers,
    how="inner",
)

number_of_candidates = (
    candidates.groupby("name").size().reindex(auxiliary["name"], fill_value=0)
)

unique_matches = candidates[candidates["name"].map(number_of_candidates).eq(1)]

print(
    "Perfiles reidentificados de forma única:",
    number_of_candidates.eq(1).sum(),
    "de",
    len(auxiliary),
)

unique_matches.head()

In [ ]:
# Paso 5: generalización de ciudad a departamento

city_to_department = {
    "Medellín": "Antioquia",
    "Bello": "Antioquia",
    "Envigado": "Antioquia",
    "Itagüí": "Antioquia",
    "Rionegro": "Antioquia",
    "Bogotá": "Bogotá D.C.",
    "Cali": "Valle del Cauca",
    "Barranquilla": "Atlántico",
    "Manizales": "Caldas",
    "Pereira": "Risaralda",
    "Cartagena": "Bolívar",
    "Bucaramanga": "Santander",
}

anonymized["department"] = anonymized["city"].map(city_to_department)

anonymized = anonymized.drop(columns=["city"])

anonymized.head()

In [ ]:
# Verificación

auxiliary_attack = auxiliary.copy()

auxiliary_attack["age_group"] = pd.cut(
    auxiliary_attack["age"],
    bins=[20, 30, 40, 50, 60, 70],
    labels=["20-29", "30-39", "40-49", "50-59", "60-69"],
    right=False,
)

auxiliary_attack["department"] = auxiliary_attack["city"].map(city_to_department)

quasi_identifiers = ["age_group", "department", "occupation"]

candidates = auxiliary_attack.merge(
    anonymized,
    on=quasi_identifiers,
    how="inner",
)

number_of_candidates = (
    candidates.groupby("name").size().reindex(auxiliary["name"], fill_value=0)
)

unique_matches = candidates[candidates["name"].map(number_of_candidates).eq(1)]

print(
    "Perfiles reidentificados de forma única:",
    number_of_candidates.eq(1).sum(),
    "de",
    len(auxiliary),
)

unique_matches.head()

In [ ]:
# Paso 6: generalización de departamento a región

department_to_region = {
    "Antioquia": "Andina",
    "Bogotá D.C.": "Andina",
    "Caldas": "Andina",
    "Risaralda": "Andina",
    "Santander": "Andina",
    "Atlántico": "Caribe",
    "Bolívar": "Caribe",
    "Valle del Cauca": "Pacífica",
}

anonymized["region"] = anonymized["department"].map(department_to_region)

anonymized = anonymized.drop(columns=["department"])

anonymized.head()

In [ ]:
# Verificación

auxiliary_attack = auxiliary.copy()

auxiliary_attack["age_group"] = pd.cut(
    auxiliary_attack["age"],
    bins=[20, 30, 40, 50, 60, 70],
    labels=["20-29", "30-39", "40-49", "50-59", "60-69"],
    right=False,
)

auxiliary_attack["department"] = auxiliary_attack["city"].map(city_to_department)

auxiliary_attack["region"] = auxiliary_attack["department"].map(department_to_region)

quasi_identifiers = ["age_group", "region", "occupation"]

candidates = auxiliary_attack.merge(
    anonymized,
    on=quasi_identifiers,
    how="inner",
)

number_of_candidates = (
    candidates.groupby("name").size().reindex(auxiliary["name"], fill_value=0)
)

unique_matches = candidates[candidates["name"].map(number_of_candidates).eq(1)]

print(
    "Perfiles reidentificados de forma única:",
    number_of_candidates.eq(1).sum(),
    "de",
    len(auxiliary),
)

unique_matches.head()

In [ ]:
# Paso 7: generalización de ocupación

occupation_to_group = {
    "Administradora": "Servicios profesionales",
    "Abogada": "Servicios profesionales",
    "Analista financiera": "Servicios profesionales",
    "Contadora": "Servicios profesionales",
    "Arquitecta": "Tecnología y diseño",
    "Diseñadora gráfica": "Tecnología y diseño",
    "Ingeniero de sistemas": "Tecnología y diseño",
    "Médico": "Salud y educación",
    "Enfermera": "Salud y educación",
    "Docente": "Salud y educación",
    "Comerciante": "Comercio y oficios",
    "Técnico electricista": "Comercio y oficios",
}

anonymized["occupation_group"] = anonymized["occupation"].map(occupation_to_group)

anonymized = anonymized.drop(columns=["occupation"])

anonymized.head()

In [ ]:
# Paso 8: se almacena el archivo

anonymized.to_csv("../submission/anonymized.csv", index=False)

In [ ]:
# Paso 9: evaluación de la utilidad analítica conservada

utility_comparison = pd.DataFrame(
    {
        "dimension": ["Edad", "Ubicación", "Ocupación"],
        "detalle_original": [
            raw["age"].nunique(),
            raw["city"].nunique(),
            raw["occupation"].nunique(),
        ],
        "detalle_anonimizado": [
            anonymized["age_group"].nunique(),
            anonymized["region"].nunique(),
            anonymized["occupation_group"].nunique(),
        ],
    }
)

utility_comparison